In [1]:
# Install from GitHub
# install.packages("remotes")
# remotes::install_github("datazoompuc/datazoom.social")

# Then load it
library(datazoom.social)
library(tidyverse)
library(readxl)

Please cite as: 
Data Zoom (2023). Data Zoom: Simplifying Access To Brazilian Microdata.
https://datazoom.com.br/en/

Or if you prefer a BibTeX entry: 
@Unpublished{DataZoom2023,
	     author = {Data Zoom},
	     title = {Data Zoom: Simplifying Access To Brazilian Microdata},
	     url = {https://datazoom.com.br/en/},
	     year = {2023},
}
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors


In [2]:
# Load PNADC data using datazoom.social
data <- load_pnadc(
  year = 2022,
  quarter = 1,
)

# Now you can view the data normally
head(data)

trying URL 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2022/PNADC_012022_20250815.zip'
Content type 'application/zip' length 202759708 bytes (193.4 MB)
downloaded 193.4 MB

trying URL 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/Documentacao/Dicionario_e_input_20221031.zip'
Content type 'application/zip' length 70208 bytes (68 KB)
downloaded 68 KB




                                                                            
1 function (..., list = character(), package = NULL, lib.loc = NULL,        
2     verbose = getOption("verbose"), envir = .GlobalEnv, overwrite = TRUE) 
3 {                                                                         
4     fileExt <- function(x) {                                              
5         db <- grepl("\\\\.[^.]+\\\\.(gz|bz2|xz)$", x)                     
6         ans <- sub(".*\\\\.", "", x)                                      

In [3]:
data <- readRDS("pnadc_2022_1.rds")

In [17]:
# Check what QUADRO 56 contains (should be government transfers)
df_transfers_detailed %>%
  filter(QUADRO == 56) %>%
  mutate(V9001 = as.numeric(V9001)) %>%
  count(V9001, sort = TRUE)

# A tibble: 4 × 2
    V9001     n
    <dbl> <int>
1 5600101 11963
2 5600201   886
3 5600401   221
4 5600301   116

What this means:

There are only 4 different income types in QUADRO 56
5600101 is by far the most common (11,963 out of ~12,600 total records = 95%)
The other 3 codes are rare (511, 82, 41 records)
Interpretation:

5600101 is almost certainly Bolsa Família (Brazil's main cash transfer program)
The other codes might be:
5600201: BPC (Benefício de Prestação Continuada - elderly/disability benefit)
5600301: Other government programs
5600401: Regional/special programs

In [18]:
# Check what QUADRO 57 contains (should be financial/rental income)
df_transfers_detailed %>%
  filter(QUADRO == 57) %>%
  mutate(V9001 = as.numeric(V9001)) %>%
  count(V9001, sort = TRUE)

# A tibble: 4 × 2
    V9001     n
    <dbl> <int>
1 5700101 10725
2 5700201   511
3 5700401    82
4 5700301    41

What this means:

There are also only 4 different income types in QUADRO 57
5700101 is the most common (10,725 records = 95%)
The other 3 codes are rare (same frequencies as QUADRO 56 - interesting!)
Interpretation:

5700101 is probably rental income (aluguel) - most common "other income"
The other codes might be:
5700201: Interest/dividends from investments
5700301: Other financial income
5700401: Remittances or other sources

# POF 2017-2018 Data Processing Pipeline

## Overview
This notebook processes Brazilian Household Budget Survey (POF) data to:
1. Load and merge multiple data sources (households, people, income, assets)
2. Separate liquid vs illiquid assets and categorize income types
3. Create demographic/economic bins for matching with PNADC data
4. Calculate Hand-to-Mouth (HTM) agent proportions within each bin

These proportions will later be applied to PNADC data via Monte Carlo simulation.

---

## Setup: Load Packages and Set Paths

In [22]:
# Load required packages
library(tidyverse)  # Data manipulation (dplyr, tidyr, readr, etc.)
library(readxl)     # Read Excel dictionary files
library(janitor)    # Clean column names

# Set file paths
data_path <- "/Users/matt/Library/CloudStorage/OneDrive-Personal/BSE/term_2/thesis/∂ata/POC/Dados_20230713" 
dict_file <- "/Users/matt/Library/CloudStorage/OneDrive-Personal/BSE/term_2/thesis/∂ata/POC/Documentacao_20230713/Dicionarios de variaveis.xls"

## Key Identifier Variables in POF Data

POF uses a hierarchical identifier system:

**Hierarchy:** UPA → Household → Consumer Unit → Person

### Variable Definitions:

- **COD_UPA** (Primary Sampling Unit): 
  - Unique identifier for each geographic sampling cluster
  - Multiple households can be in the same UPA

- **NUM_DOM** (Household Number):
  - Sequential household number within each UPA
  - `COD_UPA + NUM_DOM` = Unique household identifier

- **NUM_UC** (Consumer Unit Number):
  - Identifies consumption units within a household
  - Usually 1 (entire household is one unit)
  - Can be >1 if multiple families share one dwelling
  - `COD_UPA + NUM_DOM + NUM_UC` = Unique consumer unit identifier

- **COD_INFORMANTE** (Person Number):
  - Sequential person number within each consumer unit
  - `COD_UPA + NUM_DOM + NUM_UC + COD_INFORMANTE` = Unique person ID

---

## Helper Function: Read POF Fixed-Width Files

POF data is stored in **fixed-width format** where each variable occupies specific character positions (e.g., columns 1-5 = household ID, columns 6-10 = person ID).

The Excel dictionary files contain the layout specifications that tell us:
- Variable names
- Starting character positions  
- Width (number of characters)

This function automates reading these files correctly.

In [23]:
read_pof_table <- function(txt_filename, sheet_name_in_dict) {
  
  message(paste("Processing:", txt_filename, "..."))
  
  # Step 1: Read the data dictionary from Excel
  # The dictionary tells us where each variable starts and how wide it is
  layout <- read_excel(dict_file, sheet = sheet_name_in_dict, skip = 3) %>%
    clean_names() %>%  # Convert column names to lowercase with underscores
    
    # Filter to valid variable definitions only
    filter(!is.na(codigo_da_variavel)) %>%
    select(codigo_da_variavel, posicao_inicial, tamanho) %>%
    filter(!is.na(tamanho), !is.na(posicao_inicial)) %>%
    
    # Calculate the start and end character positions for each variable
    mutate(
      start = as.numeric(posicao_inicial),
      end = start + as.numeric(tamanho) - 1,
      col_name = codigo_da_variavel
    ) %>%
    filter(!is.na(start), !is.na(end))
  
  # Step 2: Read the fixed-width text file using the layout specifications
  df <- read_fwf(
    file = file.path(data_path, txt_filename),
    col_positions = fwf_positions(
      start = layout$start,
      end = layout$end,
      col_names = layout$col_name
    ),
    col_types = cols(.default = col_character())  # Read as character first
  )
  
  return(df)
}

## Step 1: Load Household Characteristics (DOMICILIO)

**What this table contains:**
- Household-level information: location, survey weights
- One row per household

**Key variables:**
- `COD_UPA`, `NUM_DOM`: Household identifiers
- `UF`: State code (Unidade da Federação)
- `PESO_FINAL`: Survey weight for population estimates
- `region_type`: Created from state codes

In [24]:
df_dom <- read_pof_table("DOMICILIO.txt", "Domicílio") %>%
  
  # Convert key variables from character to numeric
  mutate(across(c(COD_UPA, NUM_DOM, UF, PESO_FINAL), as.numeric)) %>%
  
  # Keep only needed variables
  select(COD_UPA, NUM_DOM, UF, PESO_FINAL) %>%
  
  # Create region variable from state codes
  # UF codes: 11-17=North, 21-29=Northeast, 31-35=Southeast, 
  #           41-43=South, 50-53=Central-West
  mutate(
    region_type = case_when(
      UF %in% c(11:17) ~ "North",
      UF %in% c(21:29) ~ "Northeast",
      UF %in% c(31:35) ~ "Southeast",
      UF %in% c(41:43) ~ "South",
      UF %in% c(50:53) ~ "Central_West",
      TRUE ~ "Other"
    )
  )

# Preview
head(df_dom)

Processing: DOMICILIO.txt ...


# A tibble: 6 × 5
    COD_UPA NUM_DOM    UF PESO_FINAL region_type
      <dbl>   <dbl> <dbl>      <dbl> <chr>      
1 110005400       1    11       373. North      
2 110005400       2    11       373. North      
3 110005400       4    11       373. North      
4 110005400       5    11       373. North      
5 110005400       6    11       373. North      
6 110005400       7    11       373. North      

## Step 2: Load Individual Demographics (MORADOR)

**What this table contains:**
- Person-level demographic information
- One row per person in each household

**Key variables:**
- `COD_UPA`, `NUM_DOM`, `NUM_UC`, `COD_INFORMANTE`: Person identifiers
- `V0403`: Age in years
- `V0404`: Sex (1=Male, 2=Female)

**TODO:** Add education variable once identified in dictionary

In [25]:
# ----------------------------------------------------------------------------
# B. MORADOR (Individual Residents)
# ----------------------------------------------------------------------------

df_mor <- read_pof_table("MORADOR.txt", "Morador") %>%
  mutate(across(c(COD_UPA, NUM_DOM, NUM_UC, COD_INFORMANTE, V0403, V0404), 
                as.numeric)) %>%
  select(COD_UPA, NUM_DOM, NUM_UC, COD_INFORMANTE, V0403, V0404) %>%
  rename(age = V0403, sex = V0404)

Processing: MORADOR.txt ...


---

## Step 3: Load Labor Income (RENDIMENTO_TRABALHO)

** What this table contains:**
- Income from work: salaries, wages, self-employment
- Multiple rows per person (people can have multiple jobs)
- Each person averages **2 records** (likely gross + deductions, or multiple jobs)

**Key variable:**
- `V8500_DEFLA`: Deflated (inflation-adjusted) labor income value

**Why we aggregate:**
- Need one total per consumer unit for household-level analysis
- Sum all income sources to get total labor income

In [26]:
# ----------------------------------------------------------------------------
# C. RENDIMENTO_TRABALHO (Labor Income)
# ----------------------------------------------------------------------------

df_inc <- read_pof_table("RENDIMENTO_TRABALHO.txt", "Rendimento do Trabalho") %>%
  mutate(across(c(COD_UPA, NUM_DOM, NUM_UC, V8500_DEFLA), as.numeric)) %>%
  group_by(COD_UPA, NUM_DOM, NUM_UC) %>%
  summarise(
    total_labor_income = sum(V8500_DEFLA, na.rm = TRUE),
    .groups = "drop"
  )

Processing: RENDIMENTO_TRABALHO.txt ...


---

## Step 4: Load Assets (INVENTARIO) - LIQUID vs ILLIQUID

**What this table contains:**
- Household assets: real estate, vehicles, financial assets, durables
- Multiple rows per household (one per asset type)
- **CRITICAL:** Must distinguish liquid from illiquid assets for HTM classification

**Asset categories by QUADRO:**
- **QUADRO 12:** Vehicles (cars, motorcycles) → **ILLIQUID**
- **QUADRO 13:** Real estate (houses, apartments, land) → **ILLIQUID**
- **QUADRO 14:** Financial assets (savings, stocks, bonds) → **LIQUID** ✓
- **QUADRO 15:** Durables (furniture, appliances) → **ILLIQUID**

**Key variable:**
- `V9001`: Current market value of the asset
- `QUADRO`: Asset type identifier

**Why this matters for HTM:**
- **W-HtM**: High illiquid assets (owns home/car) BUT low liquid assets
- **P-HtM**: Low illiquid AND low liquid assets
- **Ricardian**: High liquid assets (has savings that earn interest)

In [27]:
# ----------------------------------------------------------------------------
# D. INVENTARIO (Asset Inventory) - SEPARATE LIQUID vs ILLIQUID
# ----------------------------------------------------------------------------

df_asset_categorized <- read_pof_table("INVENTARIO.txt", "Inventário") %>%
  mutate(across(c(COD_UPA, NUM_DOM, NUM_UC, QUADRO, V9001), as.numeric)) %>%
  group_by(COD_UPA, NUM_DOM, NUM_UC) %>%
  summarise(
    # ILLIQUID ASSETS (cannot quickly convert to cash)
    total_illiquid = sum(V9001[QUADRO %in% c(12, 13, 15)], na.rm = TRUE),
    real_estate_value = sum(V9001[QUADRO == 13], na.rm = TRUE),
    vehicle_value = sum(V9001[QUADRO == 12], na.rm = TRUE),
    
    # LIQUID ASSETS (can quickly access - savings, stocks, bonds)
    total_liquid_assets = sum(V9001[QUADRO == 14], na.rm = TRUE),
    
    # Total for reference
    total_assets = sum(V9001, na.rm = TRUE),
    .groups = "drop"
  )

Processing: INVENTARIO.txt ...


---

## Step 5: Load Other Income/Transfers (OUTROS_RENDIMENTOS) - BY TYPE

**What this table contains:**
- Non-labor income: government transfers, pensions, financial income, rental income
- Multiple rows per person (people can have multiple income sources)
- Each person averages **~4 records** (e.g., pension + Bolsa Família + interest + rental)

**Income categories by QUADRO:**
- **QUADRO 54:** Other labor-related (bonuses, commissions)
- **QUADRO 55:** Pensions and retirement benefits
- **QUADRO 56:** Government transfers (Bolsa Família, BPC) → **Key for P-HtM**
- **QUADRO 57:** Financial & rental income (interest, dividends, rent) → **Key for W-HtM vs P-HtM**

**Key variable:**
- `V8500_DEFLA`: Deflated income value
- `QUADRO`: Income type identifier

**Why income composition matters:**
| Person Type | govt_transfers (Q56) | financial_rental (Q57) | assets | Classification |
|-------------|---------------------|----------------------|---------|----------------|
| Person A    | 500 (Bolsa Família) | 0                    | 0       | **P-HtM** (poor, relies on aid) |
| Person B    | 0                   | 5000 (rental)        | 500k    | **W-HtM** (owns property) |
| Person C    | 0                   | 2000 (interest)      | 10k     | **Ricardian** (has liquid savings) |

**Action:** Separate by QUADRO BEFORE aggregating (don't lump all transfers together!)

In [28]:
# ----------------------------------------------------------------------------
# E. OUTROS_RENDIMENTOS (Other Income/Transfers) - CATEGORIZED BY TYPE
# ----------------------------------------------------------------------------

df_transfers_detailed <- read_pof_table("OUTROS_RENDIMENTOS.txt", "Outros Rendimentos") %>%
  mutate(across(c(COD_UPA, NUM_DOM, NUM_UC, COD_INFORMANTE, QUADRO, V9001, V8500_DEFLA), 
                as.numeric))

# Aggregate by income type (QUADRO)
df_transfers_categorized <- df_transfers_detailed %>%
  group_by(COD_UPA, NUM_DOM, NUM_UC, COD_INFORMANTE) %>%
  summarise(
    pension_income = sum(V8500_DEFLA[QUADRO == 55], na.rm = TRUE),
    govt_transfers = sum(V8500_DEFLA[QUADRO == 56], na.rm = TRUE),      # P-HtM indicator
    financial_rental = sum(V8500_DEFLA[QUADRO == 57], na.rm = TRUE),    # W-HtM vs P-HtM
    other_labor = sum(V8500_DEFLA[QUADRO == 54], na.rm = TRUE),
    total_transfers = sum(V8500_DEFLA, na.rm = TRUE),
    .groups = "drop"
  )

Processing: OUTROS_RENDIMENTOS.txt ...


---

## Step 6: Merge All Data Tables

**What we're doing:**
- Starting with person-level data (`df_mor`)
- Adding household characteristics, income, and assets
- Result: One row per person with all financial information

**Join structure:**
1. Add household info (region, weights) by `COD_UPA + NUM_DOM`
2. Add labor income by `COD_UPA + NUM_DOM + NUM_UC`
3. Add assets (liquid & illiquid) by `COD_UPA + NUM_DOM + NUM_UC`
4. Add categorized transfers by `COD_UPA + NUM_DOM + NUM_UC + COD_INFORMANTE`

**Missing values:**
- `NA` values will appear for people without income/asset records
- We'll handle these explicitly in the next step

In [29]:
# ============================================================================
# MERGE ALL DATA TABLES
# ============================================================================

master_data <- df_mor %>%
  left_join(df_dom, by = c("COD_UPA", "NUM_DOM")) %>%
  left_join(df_inc, by = c("COD_UPA", "NUM_DOM", "NUM_UC")) %>%
  left_join(df_asset_categorized, by = c("COD_UPA", "NUM_DOM", "NUM_UC")) %>%
  left_join(df_transfers_categorized, by = c("COD_UPA", "NUM_DOM", "NUM_UC", "COD_INFORMANTE"))

---

## Step 7: Check Missing Data Patterns

**Purpose:** Understand how much missing data we have before deciding how to handle it

**Questions to answer:**
- What % of people are missing labor income data?
- What % are missing asset data?
- What % are missing transfer data?

**Decision needed:** 
- Should `NA` = 0 (no income/assets)?
- Or should we exclude people with missing data?

In [30]:
# Check missing data patterns
master_data %>%
  summarise(
    n_total = n(),
    n_missing_labor = sum(is.na(total_labor_income)),
    n_missing_assets = sum(is.na(total_illiquid)),
    n_missing_liquid_assets = sum(is.na(total_liquid_assets)),
    n_missing_transfers = sum(is.na(total_transfers)),
    pct_missing_labor = mean(is.na(total_labor_income)) * 100,
    pct_missing_assets = mean(is.na(total_illiquid)) * 100,
    pct_missing_liquid = mean(is.na(total_liquid_assets)) * 100,
    pct_missing_transfers = mean(is.na(total_transfers)) * 100
  )

# A tibble: 1 × 9
  n_total n_missing_labor n_missing_assets n_missing_liquid_assets
    <int>           <int>            <int>                   <int>
1  178431           20392              261                     261
# ℹ 5 more variables: n_missing_transfers <int>, pct_missing_labor <dbl>,
#   pct_missing_assets <dbl>, pct_missing_liquid <dbl>,
#   pct_missing_transfers <dbl>

---

## Step 8: Create Demographic and Economic Bins

**Purpose:** Create categories that exist in BOTH POF and PNADC for matching

**Bins to create:**
1. **Age groups:** Under 18, 18-24, 25-34, 35-44, 45-54, 55-64, 65+
2. **Gender:** Male, Female
3. **Labor status:** Employed, Not Employed, Out of Labor Force
4. **Income brackets:** No income, Low, Medium, High, Very High
5. **Education level:** TODO - need to identify variable

**Missing value handling:**
- Replace `NA` with 0 for income/assets
- **Rationale:** Survey records without income likely represent truly zero income

In [31]:
# ============================================================================
# CREATE DEMOGRAPHIC/ECONOMIC BINS
# ============================================================================

pof_with_bins <- master_data %>%
  
  # Replace NA with 0 for income/assets
  mutate(
    total_labor_income = replace_na(total_labor_income, 0),
    total_illiquid = replace_na(total_illiquid, 0),
    total_liquid_assets = replace_na(total_liquid_assets, 0),
    pension_income = replace_na(pension_income, 0),
    govt_transfers = replace_na(govt_transfers, 0),
    financial_rental = replace_na(financial_rental, 0),
    total_transfers = replace_na(total_transfers, 0),
    total_income = total_labor_income + total_transfers
  ) %>%
  
  # Create age brackets
  mutate(
    age_group = case_when(
      age < 18 ~ "Under 18",
      age >= 18 & age < 25 ~ "18-24",
      age >= 25 & age < 35 ~ "25-34",
      age >= 35 & age < 45 ~ "35-44",
      age >= 45 & age < 55 ~ "45-54",
      age >= 55 & age < 65 ~ "55-64",
      age >= 65 ~ "65+",
      TRUE ~ NA_character_
    )
  ) %>%
  
  # Create gender categories
  mutate(
    gender = case_when(
      sex == 1 ~ "Male",
      sex == 2 ~ "Female",
      TRUE ~ NA_character_
    )
  ) %>%
  
  # Create labor status
  mutate(
    labor_status = case_when(
      total_labor_income > 0 ~ "Employed",
      total_labor_income == 0 & age >= 18 & age < 65 ~ "Not Employed",
      age < 18 | age >= 65 ~ "Out of Labor Force",
      TRUE ~ "Not Employed"
    )
  ) %>%
  
  # Create income brackets
  mutate(
    income_bracket = case_when(
      total_income == 0 ~ "No income",
      total_income > 0 & total_income < 1000 ~ "Low (< 1k)",
      total_income >= 1000 & total_income < 3000 ~ "Medium (1-3k)",
      total_income >= 3000 & total_income < 6000 ~ "High (3-6k)",
      total_income >= 6000 ~ "Very High (6k+)",
      TRUE ~ NA_character_
    )
  )

---

## Step 9: View Bin Structure

**Purpose:** Check how people are distributed across our demographic bins

This helps us:
- Identify bins with very few people (may need to combine)
- Ensure bins are meaningful for analysis
- Verify our categorization makes sense

In [32]:
# View the bin structure
pof_with_bins %>%
  count(age_group, gender, labor_status, income_bracket) %>%
  arrange(desc(n)) %>%
  head(20)

# A tibble: 20 × 5
   age_group gender labor_status income_bracket      n
   <chr>     <chr>  <chr>        <chr>           <int>
 1 Under 18  Male   Employed     Medium (1-3k)    9812
 2 Under 18  Female Employed     Medium (1-3k)    9208
 3 Under 18  Male   Employed     Low (< 1k)       5612
 4 Under 18  Female Employed     Low (< 1k)       5223
 5 Under 18  Male   Employed     High (3-6k)      4755
 6 35-44     Male   Employed     Very High (6k+)  4488
 7 Under 18  Female Employed     High (3-6k)      4390
 8 25-34     Female Employed     Medium (1-3k)    4246
 9 25-34     Male   Employed     Very High (6k+)  4118
10 35-44     Female Employed     Medium (1-3k)    4036
11 35-44     Female Employed     Very High (6k+)  3907
12 45-54     Male   Employed     Very High (6k+)  3867
13 25-34     Female Employed     Very High (6k+)  3490
14 18-24     Female Employed     Medium (1-3k)    3452
15 45-54     Female Employed     Very High (6k+)  3397
16 45-54     Female Employed     Medium (1-3k)

# View the bin structure
pof_with_bins %>%
  count(age_group, gender, labor_status, income_bracket) %>%
  arrange(desc(n)) %>%
  head(20)

---

## Next Steps (To Be Completed)

### 1. Define HTM Classification Thresholds

Need to determine cutoffs based on:
- `govt_transfers` (QUADRO 56) - for P-HtM identification
- `total_liquid_assets` (QUADRO 14 in INVENTARIO) - key distinction
- `total_illiquid` (QUADRO 12, 13, 15 in INVENTARIO) - for W-HtM
- `financial_rental` (QUADRO 57) - indicates asset income

**Proposed criteria:**
```r
# agent_type = case_when(
#   # P-HtM: Low govt transfers, NO liquid assets, low illiquid assets
#   govt_transfers < threshold_govt &
#   total_liquid_assets < threshold_liquid &
#   total_illiquid < threshold_illiquid ~ "P-HtM",
#   
#   # W-HtM: High illiquid assets BUT low liquid assets
#   total_illiquid >= threshold_illiquid &
#   total_liquid_assets < threshold_liquid ~ "W-HtM",
#   
#   # Ricardian: Sufficient liquid assets to smooth consumption
#   total_liquid_assets >= threshold_liquid ~ "Ricardian",
#   
#   TRUE ~ "Ricardian"
# )

---

## Next Steps: HTM Classification and PNADC Application

This section outlines the remaining tasks to complete the agent type classification and apply it to PNADC data.

---

### Step 1: Explore Income and Asset Distributions

Before setting thresholds, we need to understand the data distributions.

**Questions to answer:**
- What are typical values for government transfers?
- How many households have liquid vs illiquid assets?
- What proportions would different threshold choices create?

**Run this exploration code:**

In [34]:
# Explore distributions to inform threshold decisions
pof_with_bins %>%
  summarise(
    # Government transfers (QUADRO 56)
    govt_transfers_p10 = quantile(govt_transfers, 0.10),
    govt_transfers_p25 = quantile(govt_transfers, 0.25),
    govt_transfers_p50 = quantile(govt_transfers, 0.50),
    govt_transfers_mean = mean(govt_transfers[govt_transfers > 0]),
    pct_receiving_govt = mean(govt_transfers > 0) * 100,
    
    # Liquid assets (QUADRO 14)
    liquid_p10 = quantile(total_liquid_assets, 0.10),
    liquid_p25 = quantile(total_liquid_assets, 0.25),
    liquid_p50 = quantile(total_liquid_assets, 0.50),
    liquid_mean = mean(total_liquid_assets[total_liquid_assets > 0]),
    pct_has_liquid = mean(total_liquid_assets > 0) * 100,
    
    # Illiquid assets (QUADRO 12, 13, 15)
    illiquid_p10 = quantile(total_illiquid, 0.10),
    illiquid_p25 = quantile(total_illiquid, 0.25),
    illiquid_p50 = quantile(total_illiquid, 0.50),
    illiquid_mean = mean(total_illiquid[total_illiquid > 0]),
    pct_has_illiquid = mean(total_illiquid > 0) * 100,
    
    # Financial/rental income (QUADRO 57)
    financial_p10 = quantile(financial_rental, 0.10),
    financial_p25 = quantile(financial_rental, 0.25),
    financial_p50 = quantile(financial_rental, 0.50),
    financial_mean = mean(financial_rental[financial_rental > 0]),
    pct_has_financial = mean(financial_rental > 0) * 100
  )

# A tibble: 1 × 20
  govt_transfers_p10 govt_transfers_p25 govt_transfers_p50 govt_transfers_mean
               <dbl>              <dbl>              <dbl>               <dbl>
1                  0                  0                  0              11903.
# ℹ 16 more variables: pct_receiving_govt <dbl>, liquid_p10 <dbl>,
#   liquid_p25 <dbl>, liquid_p50 <dbl>, liquid_mean <dbl>,
#   pct_has_liquid <dbl>, illiquid_p10 <dbl>, illiquid_p25 <dbl>,
#   illiquid_p50 <dbl>, illiquid_mean <dbl>, pct_has_illiquid <dbl>,
#   financial_p10 <dbl>, financial_p25 <dbl>, financial_p50 <dbl>,
#   financial_mean <dbl>, pct_has_financial <dbl>

---

### Step 2: Define HTM Classification Criteria

Based on the distributions above, define thresholds that create meaningful agent type proportions.

**Classification logic:**

| Agent Type | Liquid Assets | Illiquid Assets | Govt Transfers | Financial Income |
|------------|---------------|-----------------|----------------|------------------|
| **P-HtM** (Poor) | Very Low | Very Low | Low/Medium | None/Very Low |
| **W-HtM** (Wealthy) | Very Low | **High** | Any | Low (from illiquid assets) |
| **Ricardian** | **High** | Any | Any | Any |

**Key insight:** The distinction between W-HtM and P-HtM is:
- **W-HtM** owns valuable illiquid assets (home, car) but lacks liquid savings
- **P-HtM** has neither liquid nor illiquid assets

**Implement classification:**

In [35]:
# Define thresholds (adjust based on distributions above)
threshold_liquid <- 1000      # Minimum liquid assets for Ricardian
threshold_illiquid <- 50000   # Minimum illiquid assets for W-HtM
threshold_govt <- 500         # Maximum govt transfers for P-HtM

# Create HTM classification
pof_with_htm <- pof_with_bins %>%
  mutate(
    agent_type = case_when(
      # Ricardian: Has sufficient liquid assets to smooth consumption
      total_liquid_assets >= threshold_liquid ~ "Ricardian",
      
      # W-HtM: High illiquid assets BUT low liquid assets
      # Asset-rich (owns home/car) but cash-poor
      total_illiquid >= threshold_illiquid & 
      total_liquid_assets < threshold_liquid ~ "W-HtM",
      
      # P-HtM: Low liquid assets, low illiquid assets
      # Truly poor and liquidity constrained
      total_liquid_assets < threshold_liquid & 
      total_illiquid < threshold_illiquid ~ "P-HtM",
      
      # Default to Ricardian (middle class with some resources)
      TRUE ~ "Ricardian"
    )
  )

# Check the distribution
pof_with_htm %>%
  count(agent_type) %>%
  mutate(percentage = n / sum(n) * 100)

# A tibble: 2 × 3
  agent_type      n percentage
  <chr>       <int>      <dbl>
1 P-HtM         261      0.146
2 Ricardian  178170     99.9  

---

### Step 3: Verify Classification by Region

Check if HTM proportions vary by region as expected (e.g., more P-HtM in North/Northeast, more W-HtM in South/Southeast).

In [36]:
# HTM distribution by region
pof_with_htm %>%
  group_by(region_type, agent_type) %>%
  summarise(
    n = n(),
    avg_liquid = mean(total_liquid_assets),
    avg_illiquid = mean(total_illiquid),
    avg_govt_transfers = mean(govt_transfers),
    avg_financial = mean(financial_rental),
    .groups = "drop"
  ) %>%
  arrange(region_type, agent_type)

# A tibble: 10 × 7
   region_type  agent_type     n avg_liquid avg_illiquid avg_govt_transfers
   <chr>        <chr>      <int>      <dbl>        <dbl>              <dbl>
 1 Central_West P-HtM         11         0             0           12458.  
 2 Central_West Ricardian  21326  23470304.            0            1361.  
 3 North        P-HtM        110         0             0             228.  
 4 North        Ricardian  29381  17782387.            0             215.  
 5 Northeast    P-HtM         46         0             0               0   
 6 Northeast    Ricardian  60748  20154306.            0             415.  
 7 South        P-HtM         50         0             0               5.35
 8 South        Ricardian  23355  24480867.            0            1429.  
 9 Southeast    P-HtM         44         0             0              93.7 
10 Southeast    Ricardian  43360  22812347.            0            1353.  
# ℹ 1 more variable: avg_financial <dbl>

---

### Step 4: Calculate Bin Proportions for PNADC Matching

For each demographic bin (age × gender × education × labor status), calculate the proportion of each agent type.

**Purpose:** These proportions will be used to randomly assign agent types to PNADC individuals via Monte Carlo simulation.

**Note:** Education variable must be added before this step!

In [37]:
# Calculate HTM proportions within each demographic bin
bin_proportions <- pof_with_htm %>%
  group_by(age_group, gender, labor_status) %>%  # Add education when available
  summarise(
    n_total = n(),
    n_phtm = sum(agent_type == "P-HtM"),
    n_whtm = sum(agent_type == "W-HtM"),
    n_ricardian = sum(agent_type == "Ricardian"),
    prop_phtm = mean(agent_type == "P-HtM"),
    prop_whtm = mean(agent_type == "W-HtM"),
    prop_ricardian = mean(agent_type == "Ricardian"),
    .groups = "drop"
  ) %>%
  # Filter out bins with very few observations
  filter(n_total >= 10)

# View some examples
bin_proportions %>%
  arrange(desc(n_total)) %>%
  head(20)

# A tibble: 20 × 10
   age_group gender labor_status     n_total n_phtm n_whtm n_ricardian prop_phtm
   <chr>     <chr>  <chr>              <int>  <int>  <int>       <int>     <dbl>
 1 Under 18  Male   Employed           22585     29      0       22556  0.00128 
 2 Under 18  Female Employed           21034     32      0       21002  0.00152 
 3 35-44     Female Employed           12810     15      0       12795  0.00117 
 4 25-34     Female Employed           12399     18      0       12381  0.00145 
 5 35-44     Male   Employed           11994     16      0       11978  0.00133 
 6 25-34     Male   Employed           11751     15      0       11736  0.00128 
 7 45-54     Female Employed           11050     13      0       11037  0.00118 
 8 45-54     Male   Employed           10223     19      0       10204  0.00186 
 9 18-24     Male   Employed            9566     22      0        9544  0.00230 
10 18-24     Female Employed            8956     11      0        8945  0.00123 
11 55-64

---

### Step 5: Apply to PNADC via Monte Carlo Simulation

**Workflow:**
1. Load PNADC data with same demographic variables
2. Create identical bins (age_group, gender, education, labor_status)
3. Match each PNADC person to their corresponding POF bin
4. Randomly assign agent type using the proportions from that bin

**Example Monte Carlo assignment:**

In [38]:
# # Example: Apply to PNADC (once PNADC data is loaded)
# 
# pnadc_with_agent_type <- pnadc_data %>%
#   
#   # Create same bins as POF
#   mutate(
#     age_group = case_when(...),  # Same logic as POF
#     gender = case_when(...),
#     labor_status = case_when(...)
#   ) %>%
#   
#   # Join with POF bin proportions
#   left_join(
#     bin_proportions, 
#     by = c("age_group", "gender", "labor_status")  # Add education
#   ) %>%
#   
#   # Monte Carlo assignment
#   # Set seed for reproducibility
#   mutate(
#     random_draw = runif(n()),
#     agent_type = case_when(
#       random_draw <= prop_phtm ~ "P-HtM",
#       random_draw <= prop_phtm + prop_whtm ~ "W-HtM",
#       TRUE ~ "Ricardian"
#     )
#   ) %>%
#   
#   # Remove temporary columns
#   select(-random_draw, -starts_with("prop_"), -starts_with("n_"))

---

### Outstanding Tasks Checklist

**Before HTM classification:**
- [ ] Identify education variable in POF MORADOR table
- [ ] Verify QUADRO 14 exists in INVENTARIO (run exploration)
- [ ] Run distribution analysis to set meaningful thresholds
- [ ] Ensure bins create balanced groups (not too small)

**For PNADC matching:**
- [ ] Load PNADC data with demographics and income variables
- [ ] Create identical bins in PNADC (verify variable compatibility)
- [ ] Check for bins that exist in PNADC but not POF (handle missing matches)
- [ ] Set random seed before Monte Carlo for reproducibility
- [ ] Validate: Do assigned proportions match POF proportions?

**Data quality checks:**
- [ ] Compare POF vs PNADC sample sizes by bin
- [ ] Check for bins with very few observations (< 10 people)
- [ ] Verify income/asset distributions are reasonable
- [ ] Cross-validate: Do regional patterns make sense?

---

## Save Processed Data

Save the POF data with HTM classifications and bin proportions for future use.

In [39]:
# Save POF data with bins and agent types (once classification is complete)
# saveRDS(pof_with_htm, "pof_with_htm_classification.rds")

# Save bin proportions for PNADC application
# saveRDS(bin_proportions, "pof_bin_proportions.rds")

# Optional: Save summary statistics
# summary_stats <- pof_with_htm %>%
#   group_by(region_type, agent_type) %>%
#   summarise(across(c(total_income, total_liquid_assets, total_illiquid), 
#                    list(mean = mean, median = median, sd = sd)))
# saveRDS(summary_stats, "pof_htm_summary_stats.rds")